# <font color="#418FDE" size="6.5" uppercase>**Tree Regularization**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Explain CART splitting and impurity reduction at a practical level. 
- Diagnose overfitting and instability in deep decision trees. 
- Apply pruning and structural constraints to improve generalization. 


## **1. CART Splitting Basics**

### **1.1. Split Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_01_01.jpg?v=1783848994" width="250">



>* Tests feature rules to split data better.
>* Greedily picks the most homogeneous split.

>* Trees split mixed data into cleaner groups.
>* Best split changes by node context.

>* Binary splits simplify trees and improve separation.
>* Best split asks the most useful next question.



In [ ]:
#@title Python Code - Split Selection

# This script shows greedy CART split selection.
# We compare candidate thresholds using Gini impurity.
# Small output highlights local impurity reduction clearly.

import numpy as np
import pandas as pd

# Build a tiny loan style dataset.
data = pd.DataFrame({
    "income": [30, 35, 40, 45, 60, 65, 70, 90],
    "defaulted": [1, 1, 1, 0, 0, 0, 0, 0],
})

# Set a deterministic display order.
data = data.sort_values("income").reset_index(drop=True)

# Define Gini impurity for binary labels.
def gini(labels):
    counts = labels.value_counts(normalize=True)
    return 1 - np.sum(counts.to_numpy() ** 2)

# Compute weighted child impurity safely.
def split_score(frame, threshold):
    left = frame[frame["income"] <= threshold]
    right = frame[frame["income"] > threshold]
    if len(left) == 0 or len(right) == 0:
        return None
    left_gini = gini(left["defaulted"])
    right_gini = gini(right["defaulted"])
    weighted = (
        len(left) * left_gini + len(right) * right_gini
    ) / len(frame)
    return weighted, len(left), len(right)

# Create candidate thresholds between incomes.
values = data["income"].to_numpy()
thresholds = (values[:-1] + values[1:]) / 2

# Measure the parent node impurity.
parent_gini = gini(data["defaulted"])
results = []

# Evaluate each possible binary split.
for threshold in thresholds:
    score = split_score(data, threshold)
    if score is not None:
        weighted, left_n, right_n = score
        gain = parent_gini - weighted
        results.append((threshold, weighted, gain, left_n, right_n))

# Pick the greedy best split.
best = min(results, key=lambda row: row[1])

# Print a compact teaching summary.
print("Parent Gini:", round(parent_gini, 3))
print("Candidate thresholds:", [float(x) for x in thresholds])
print("Best threshold:", best[0])
print("Best child Gini:", round(best[1], 3))
print("Impurity reduction:", round(best[2], 3))
print("Left and right sizes:", best[3], best[4])
print("Greedy choice: ask income <= threshold next.")



### **1.2. Impurity Reduction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_01_02.jpg?v=1783848992" width="250">



>* Impure nodes contain mixed outcome examples.
>* Good splits create clearer, more predictable groups.

>* Judge splits by weighted impurity reduction.
>* Tiny pure groups may help little.

>* CART greedily picks the best local split.
>* Best splits create cleaner, more predictable groups.



In [ ]:
#@title Python Code - Impurity Reduction

# This script demonstrates impurity reduction clearly.
# We compare two possible CART splits.
# Results show weighted impurity improvement.

import numpy as np

# Use a deterministic tiny dataset.
y = np.array([0, 0, 0, 1, 1, 1, 1, 0])
X = np.array([2, 3, 4, 5, 6, 7, 8, 9])

# Define a simple Gini impurity function.
def gini(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()

    return 1.0 - np.sum(probs ** 2)

# Compute weighted child impurity safely.
def split_score(feature, labels, threshold):
    left_mask = feature <= threshold
    right_mask = feature > threshold

    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return None

    left_y = labels[left_mask]
    right_y = labels[right_mask]
    total = len(labels)

    weighted = (
        len(left_y) / total * gini(left_y)
        + len(right_y) / total * gini(right_y)
    )

    reduction = gini(labels) - weighted
    return weighted, reduction, left_y, right_y

# Measure impurity before any split.
parent_gini = gini(y)
score_a = split_score(X, y, 4)
score_b = split_score(X, y, 6)

# Print a short practical comparison.
print(f"Parent Gini: {parent_gini:.3f}")
print("Split A: X <= 4")
print(f"Weighted child Gini: {score_a[0]:.3f}")
print(f"Impurity reduction: {score_a[1]:.3f}")

print("Split B: X <= 6")
print(f"Weighted child Gini: {score_b[0]:.3f}")
print(f"Impurity reduction: {score_b[1]:.3f}")

# Explain which split CART would prefer.
better = "A" if score_a[1] > score_b[1] else "B"
print(f"Better split here: {better}")
print("CART prefers the larger impurity reduction.")



### **1.3. Choosing Best Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_01_03.jpg?v=1783848990" width="250">



>* Best split makes child groups more similar.
>* CART tests many splits for purity gain.

>* Compare splits by weighted impurity reduction.
>* Best splits create cleaner, sizable groups.

>* Best split is the best local choice.
>* Greedy splits can be unstable, yet interpretable.



In [ ]:
#@title Python Code - Choosing Best Split

# This script compares simple CART split choices.
# It shows weighted impurity reduction clearly.
# Small data keeps the example beginner friendly.

import numpy as np
import pandas as pd

# Build a tiny binary classification dataset.
data = pd.DataFrame({
    "debt_ratio": [20, 25, 30, 35, 40, 60, 65, 70],
    "default":    [0, 0, 0, 1, 1, 1, 1, 1],
})

# Define a simple Gini impurity function.
def gini(labels):
    counts = np.bincount(labels, minlength=2)
    probs = counts / counts.sum()
    return 1.0 - np.sum(probs ** 2)

# Compute weighted child impurity after a split.
def split_score(frame, threshold):
    left = frame[frame["debt_ratio"] <= threshold]
    right = frame[frame["debt_ratio"] > threshold]
    if len(left) == 0 or len(right) == 0:
        return None

    left_gini = gini(left["default"].to_numpy())
    right_gini = gini(right["default"].to_numpy())
    weighted = (
        len(left) / len(frame) * left_gini +
        len(right) / len(frame) * right_gini
    )
    return left, right, left_gini, right_gini, weighted

# Measure impurity before any split.
parent_gini = gini(data["default"].to_numpy())
thresholds = [32.5, 50.0, 67.5]
results = []

# Evaluate a few candidate split points.
for threshold in thresholds:
    outcome = split_score(data, threshold)
    if outcome is not None:
        left, right, left_gini, right_gini, weighted = outcome
        gain = parent_gini - weighted
        results.append((threshold, len(left), len(right), weighted, gain))

# Choose the split with largest impurity reduction.
best = max(results, key=lambda row: row[4])

# Print a compact teaching summary.
print(f"Parent Gini: {parent_gini:.3f}")
for threshold, left_n, right_n, weighted, gain in results:
    print(
        f"Split at {threshold:>4}: "
        f"left={left_n}, right={right_n}, "
        f"weighted Gini={weighted:.3f}, gain={gain:.3f}"
    )

# State the best local CART choice.
print(f"Best split: debt_ratio <= {best[0]}")



## **2. Controlling Tree Overfit**

### **2.1. Signs of Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_02_01.jpg?v=1783848999" width="250">



>* High training accuracy, weak test performance.
>* Tree memorizes noise instead of patterns.

>* Deep branches create tiny, unreliable leaf groups.
>* Complex trees often learn noise, not patterns.

>* Overfit trees chase minor, hard-to-explain details.
>* Precise-looking rules often fail on new data.



### **2.2. Signs of Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_02_02.jpg?v=1783848997" width="250">



>* Excellent training results, poor new-data performance.
>* Tiny specific splits learn noise, not patterns.

>* Overfit trees grow deep, fragmented branches.
>* Tiny groups create unreliable, unjustified rules.

>* Overfit trees change across samples or folds.
>* Training confidence fails to generalize reliably.



### **2.3. Tree Instability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_02_03.jpg?v=1783849004" width="250">



>* Small data changes can reshape deep trees
>* This fragility hurts reliable new predictions

>* Small data changes can reshape tree splits.
>* This hurts interpretation, trust, and reliability.

>* Deep trees overfit isolated, nonrepeatable cases.
>* Instability shows brittle patterns and unreliable predictions.



In [ ]:
#@title Python Code - Tree Instability

# Small samples can reshape deep trees quickly.
# This script illustrates instability without machine learning.
# We compare two hand built split rules.

import numpy as np
import matplotlib.pyplot as plt

# Use a fixed seed for reproducibility.
rng = np.random.default_rng(7)
# Create one feature and binary labels.
x = np.linspace(0.0, 10.0, 24)

y = (x > 5.0).astype(int)
# Add a few borderline noisy cases.
flip_ids = np.array([10, 11, 12, 13])
y[flip_ids] = 1 - y[flip_ids]

# Build a tiny second sample.
keep = np.ones(x.size, dtype=bool)
# Remove two borderline observations.
keep[[11, 12]] = False
x2 = x[keep]

y2 = y[keep]
# Search candidate split thresholds.
thresholds = (x[:-1] + x[1:]) / 2

# Define Gini impurity for labels.
def gini(labels):
    if labels.size == 0:
        return 0.0

    p = labels.mean()
    return 1.0 - p**2 - (1.0 - p)**2

# Score one split by impurity reduction.
def best_split(values, labels, cuts):
    base = gini(labels)
    best_gain = -1.0

    best_cut = None
    for cut in cuts:
        left = labels[values <= cut]
        right = labels[values > cut]

        weight = left.size / labels.size
        child = weight * gini(left)
        child += (1.0 - weight) * gini(right)

        gain = base - child
        if gain > best_gain:
            best_gain = gain
            best_cut = cut

    return best_cut, best_gain

# Find best first split twice.
cut1, gain1 = best_split(x, y, thresholds)
cut2, gain2 = best_split(x2, y2, thresholds[:-1])

# Print a short diagnosis.
print(f"Sample A best split: x <= {cut1:.2f}")
print(f"Sample B best split: x <= {cut2:.2f}")
print(f"Gain A: {gain1:.3f}, Gain B: {gain2:.3f}")
print("Small data changes moved the first split.")

# Plot both samples and split locations.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x, y, s=45, color="tab:blue")
axes[0].axvline(cut1, color="tab:red", linestyle="--")

axes[0].set_title("Sample A")
axes[0].set_xlabel("Feature value")
axes[0].set_ylabel("Class label")

axes[1].scatter(x2, y2, s=45, color="tab:green")
axes[1].axvline(cut2, color="tab:red", linestyle="--")
axes[1].set_title("Sample B")

axes[1].set_xlabel("Feature value")
plt.tight_layout()
plt.show()



## **3. Pruning and Constraints**

### **3.1. Constraint Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_03_01.jpg?v=1783849000" width="250">



>* Constraints limit tree complexity during training.
>* They reduce overfitting and improve generalization.

>* Constraints balance underfitting against overfitting.
>* Choose settings that validate and stay stable.

>* Constraints improve clarity and trust in rules.
>* They reduce fragile splits and aid generalization.



### **3.2. Pruning Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_03_02.jpg?v=1783849002" width="250">



>* Pruning removes weak branches for generalization.
>* It reduces overfitting and improves interpretability.

>* Compare tree sizes for best performance.
>* Remove weak splits that add little.

>* Pruning reduces overfitting and tree instability.
>* Smaller trees are clearer and more reliable.



### **3.3. Minimum Samples Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_03_03.jpg?v=1783849006" width="250">



>* Sets sample threshold before splitting nodes.
>* Prevents overfitting by requiring stronger evidence.

>* Small subgroup splits can overfit badly.
>* Higher thresholds require stronger evidence.

>* Balance split threshold to avoid overfitting, underfitting.
>* Tune with data size and other constraints.



In [ ]:
#@title Python Code - Minimum Samples Split

# This script illustrates minimum samples split.
# We compare simple recursive tree growth.
# Small nodes create unstable extra branches.

import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create one feature and labels.
x = np.linspace(0.0, 12.0, 24)
y = (x > 6.0).astype(int)

# Flip two labels as noise.
y[[5, 18]] = 1 - y[[5, 18]]

# Define Gini impurity helper.
def gini(labels):
    counts = np.bincount(labels, minlength=2)
    probs = counts / counts.sum()
    return 1.0 - np.sum(probs ** 2)

# Find the best split safely.
def best_split(values, labels):
    if len(values) < 2:
        return None, 0.0
    order = np.argsort(values)
    xs = values[order]
    ys = labels[order]
    parent = gini(ys)
    best_threshold, best_gain = None, 0.0

    # Test midpoints between neighbors.
    for i in range(1, len(xs)):
        if xs[i] == xs[i - 1]:
            continue
        threshold = (xs[i] + xs[i - 1]) / 2.0
        left = ys[xs <= threshold]
        right = ys[xs > threshold]

        # Skip empty child nodes.
        if len(left) == 0 or len(right) == 0:
            continue
        weighted = (
            len(left) * gini(left)
            + len(right) * gini(right)
        ) / len(ys)
        gain = parent - weighted

        # Keep the strongest split.
        if gain > best_gain:
            best_threshold, best_gain = threshold, gain
    return best_threshold, best_gain

# Grow a tiny teaching tree.
def grow(values, labels, min_split, depth=0):
    threshold, gain = best_split(values, labels)
    node = {"size": len(labels), "depth": depth}
    if threshold is None or gain <= 0.0:
        node["leaves"] = 1
        return node

    # Stop when node is too small.
    if len(labels) < min_split:
        node["leaves"] = 1
        return node
    left_mask = values <= threshold
    right_mask = ~left_mask
    left = grow(values[left_mask], labels[left_mask], min_split, depth + 1)
    right = grow(values[right_mask], labels[right_mask], min_split, depth + 1)
    node["leaves"] = left["leaves"] + right["leaves"]
    return node

# Summarize tree size recursively.
def summarize(node):
    return node["leaves"], node["depth"]

# Build two trees with constraints.
small_rule = grow(x, y, min_split=2)
large_rule = grow(x, y, min_split=8)

# Compute a useful root split.
root_threshold, root_gain = best_split(x, y)

# Print a compact teaching summary.
print(f"Samples: {len(x)}, noisy labels: 2")
print(f"Best root threshold: {root_threshold:.1f}")
print(f"Root impurity reduction: {root_gain:.3f}")
print(f"Leaves with min_split=2: {summarize(small_rule)[0]}")
print(f"Leaves with min_split=8: {summarize(large_rule)[0]}")
print("Higher minimum samples split blocks tiny branches.")
print("That usually reduces overfitting and instability.")



# <font color="#418FDE" size="6.5" uppercase>**Tree Regularization**</font>


In this lecture, you learned to:
- Explain CART splitting and impurity reduction at a practical level. 
- Diagnose overfitting and instability in deep decision trees. 
- Apply pruning and structural constraints to improve generalization. 

In the next Module (Module 17), we will go over 'None'